#### 5.1
Normal Simulation, PD Input 0 mean - 100,000 simulations, compare input vs output covariance, using test5_1.csv and testout_5.1.csv

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.correlation_tools import cov_nearest, corr_nearest

In [2]:
# read input data
data1 = pd.read_csv('../testfiles/data/test5_1.csv')
data1

,x1,x2,x3,x4,x5
0,0.084979,0.087586,0.042304,0.008984,0.003876
1,0.087586,0.160485,0.058136,0.012345,0.005326
2,0.042304,0.058136,0.037440,0.005963,0.002573
3,0.008984,0.012345,0.005963,0.001688,0.000546
4,0.003876,0.005326,0.002573,0.000546,0.000314


In [3]:
# check eigenvalue to confirm matrix is PD
eigvals = np.linalg.eigvalsh(data1)

tol = 1e-10
if np.all(eigvals > tol):
    print("Positive Definite")
elif np.all(eigvals >= -tol):
    print("Positive Semi-Definite")
else:
    print("Indefinite")

Positive Definite


In [4]:
def normal_sim(data):
    # mean vector
    n = len(data)
    mu = np.zeros(n)
    
    np.random.seed(42)
    n_sims = 100_000

    # if Cholesky doesn't fail, also confirm PD / PSD
    L = np.linalg.cholesky(data)
    
    # draw random num from normal dist, 100,000 x 5
    z = np.random.randn(n_sims, n)
    
    # correlated simulations
    x = z @ L.T

    return x

In [83]:
X = normal_sim(data1)

sim_cov = np.cov(X, rowvar=False, ddof=0)
pd.DataFrame(sim_cov)

,0,1,2,3,4
0,0.085296,0.087589,0.042489,0.009004,0.003897
1,0.087589,0.160056,0.057990,0.012316,0.005323
2,0.042489,0.057990,0.037510,0.005956,0.002581
3,0.009004,0.012316,0.005956,0.001689,0.000547
4,0.003897,0.005323,0.002581,0.000547,0.000315


#### 5.2
Normal Simulation, PSD Input 0 mean - 100,000 simulations, compare input vs output covariance using test5_2.csv and testout_5.2.csv

In [5]:
# read input data
data2 = pd.read_csv('../testfiles/data/test5_2.csv')
data2

,x1,x2,x3,x4,x5
0,0.084979,0.116781,0.042304,0.008984,0.003876
1,0.116781,0.160485,0.058136,0.012345,0.005326
2,0.042304,0.058136,0.037440,0.005963,0.002573
3,0.008984,0.012345,0.005963,0.001688,0.000546
4,0.003876,0.005326,0.002573,0.000546,0.000314


In [6]:
# check eigenvalue to confirm matrix is PSD
eigvals = np.linalg.eigvalsh(data2)

tol = 1e-10
if np.all(eigvals > tol):
    print("Positive Definite")
elif np.all(eigvals >= -tol):
    print("Positive Semi-Definite")
else:
    print("Indefinite")

Positive Semi-Definite


In [18]:
def normal_sim_svd(data):
    n = data.shape[0]
    n_sims = 100_000
    np.random.seed(42)

    U, s, Vt = np.linalg.svd(data)

    # clip small negatives (numerical issue)
    s[s < 0] = 0

    L = U @ np.diag(np.sqrt(s))

    z = np.random.randn(n_sims, n)
    x = z @ L.T

    return x

In [19]:
X = normal_sim_svd(data2)

sim_cov = np.cov(X, rowvar=False, ddof=0)
pd.DataFrame(sim_cov)

/var/folders/nm/dqpv5q9j19nbryfplm0vx2hc0000gn/T/ipykernel_13060/343555389.py:14: RuntimeWarning: divide by zero encountered in matmul
  x = z @ L.T
/var/folders/nm/dqpv5q9j19nbryfplm0vx2hc0000gn/T/ipykernel_13060/343555389.py:14: RuntimeWarning: overflow encountered in matmul
  x = z @ L.T
/var/folders/nm/dqpv5q9j19nbryfplm0vx2hc0000gn/T/ipykernel_13060/343555389.py:14: RuntimeWarning: invalid value encountered in matmul
  x = z @ L.T


,0,1,2,3,4
0,0.085241,0.117141,0.042606,0.009010,0.003894
1,0.117141,0.160979,0.058550,0.012382,0.005351
2,0.042606,0.058550,0.037666,0.005977,0.002586
3,0.009010,0.012382,0.005977,0.001690,0.000546
4,0.003894,0.005351,0.002586,0.000546,0.000315


#### 5.3
Normal Simulation, non PSD Input, 0 mean, near_psd fix - 100,000 simulations, compare input vs output covariance, using test5_3.csv and testout_5.3.csv

In [11]:
# read input data
data3 = pd.read_csv('../testfiles/data/test5_3.csv')
data3

,x1,x2,x3,x4,x5
0,0.084979,0.000000,0.042304,0.008984,0.003876
1,0.000000,0.160485,0.058136,0.012345,0.005326
2,0.042304,0.058136,0.037440,0.005963,0.002573
3,0.008984,0.012345,0.005963,0.001688,0.000546
4,0.003876,0.005326,0.002573,0.000546,0.000314


In [12]:
# check eigenvalue to confirm matrix is non-PSD
eigvals = np.linalg.eigvalsh(data3)

tol = 1e-10
if np.all(eigvals > tol):
    print("Positive Definite")
elif np.all(eigvals >= -tol):
    print("Positive Semi-Definite")
else:
    print("Indefinite")

Indefinite


In [89]:
# convert non-PSD cov to near PSD cov
def spectral_cov(cov, eps=1e-8):
    D = np.sqrt(np.diag(cov))
    corr = cov / np.outer(D, D)
    
    # eigen-decomposition
    eigval, eigvec = np.linalg.eigh(corr)

    # clip eigenvalues
    eigval_clipped = np.maximum(eigval, eps)

    # reconstruct corr matrix
    corr_psd = eigvec @ np.diag(eigval_clipped) @ eigvec.T

    # normalize
    corr_psd = corr_psd / np.outer(np.sqrt(np.diag(corr_psd)), np.sqrt(np.diag(corr_psd)))

    # corr to cov
    cov_psd = np.outer(D, D) * corr_psd
    
    return cov_psd

In [90]:
near_psd_cov = spectral_cov(data3)
near_psd_cov

array([[0.08497905, 0.00872238, 0.03783449, 0.00803433, 0.00346641],
       [0.00872238, 0.16048451, 0.05199344, 0.01104105, 0.00476366],
       [0.03783449, 0.05199344, 0.03744009, 0.00601008, 0.00259305],
       [0.00803433, 0.01104105, 0.00601008, 0.00168834, 0.00055065],
       [0.00346641, 0.00476366, 0.00259305, 0.00055065, 0.00031428]])

In [91]:
# check eigenvalue to confirm Higham cov is PSD
S = (near_psd_cov + near_psd_cov.T) / 2
eigen_val = np.linalg.eigvalsh(near_psd_cov)
eigen_val

array([2.43714352e-11, 2.39247562e-04, 4.94630865e-03, 9.41144427e-02,
       1.85606285e-01])

In [92]:
X = normal_sim(near_psd_cov)

sim_cov = np.cov(X, rowvar=False, ddof=0)
pd.DataFrame(sim_cov)

,0,1,2,3,4
0,0.085296,0.008267,0.037891,0.008023,0.003464
1,0.008267,0.160166,0.051558,0.010994,0.004746
2,0.037891,0.051558,0.037328,0.005981,0.002586
3,0.008023,0.010994,0.005981,0.001687,0.000549
4,0.003464,0.004746,0.002586,0.000549,0.000314


#### 5.4
Normal Simulation, PSD Input, 0 mean, higham fix - 100,000 simulations, compare input vs output covariance, using test5_3.csv and testout_5.4.csv

In [93]:
# convert non-PSD cov to Higham cov
higham_cov = cov_nearest(data3, method="higham")
higham_cov

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/stats/correlation_tools.py:89: IterationLimitWarning: 
Maximum iteration reached.

  warnings.warn(iteration_limit_doc, IterationLimitWarning)


array([[0.08497905, 0.0131214 , 0.03889666, 0.00825989, 0.00356373],
       [0.0131214 , 0.16048451, 0.05345311, 0.01135102, 0.0048974 ],
       [0.03889666, 0.05345311, 0.03744009, 0.00622121, 0.00268414],
       [0.00825989, 0.01135102, 0.00622121, 0.00168834, 0.00056999],
       [0.00356373, 0.0048974 , 0.00268414, 0.00056999, 0.00031428]])

In [94]:
# check eigenvalue to confirm Higham cov is PSD
S = (higham_cov + higham_cov.T) / 2
eigen_val = np.linalg.eigvalsh(higham_cov)
eigen_val

array([1.62278211e-18, 2.13259264e-04, 4.42479735e-03, 9.14958859e-02,
       1.88772342e-01])

In [95]:
X = normal_sim(higham_cov)

sim_cov = np.cov(X, rowvar=False, ddof=0)
pd.DataFrame(sim_cov)

,0,1,2,3,4
0,0.085296,0.012684,0.038956,0.008251,0.003562
1,0.012684,0.160121,0.053026,0.011302,0.004878
2,0.038956,0.053026,0.037333,0.006193,0.002677
3,0.008251,0.011302,0.006193,0.001686,0.000568
4,0.003562,0.004878,0.002677,0.000568,0.000314


#### 5.5
PCA Simulation, 99% explained, 0 mean - 100,000 simulations compare input vs output covariance, using test5_2.csv and testout_5.5.csv

In [96]:
# eigen decompose
eigen_val, eigen_vec = np.linalg.eigh(data2)

# sort descending
idx = np.argsort(eigen_val[::-1])
eigen_val = eigen_val[idx]
eigen_vec = eigen_vec[:, idx]

In [97]:
# variance explained
total_var = eigen_val.sum()
pct_explained = eigen_val / total_var
cum_explained = np.cumsum(pct_explained)

In [98]:
# choose k to reach 99% explained
threshold = 0.99
k = int(np.searchsorted(cum_explained, threshold) + 1)
k

2

In [99]:
# top k eigenvectors and eigenvalues
k_vec = eigen_vec[:, :k]
k_val = eigen_val[:k]

In [100]:
# PCA simulation
np.random.seed(42)
n_sims = 100_000
    
# draw random num from normal dist
z = np.random.randn(n_sims, k)

S = z * np.sqrt(k_val[None, :])
    
# correlated simulations
x = S @ k_vec.T

In [101]:
sim_cov = np.cov(x, rowvar=False, ddof=0)
pd.DataFrame(sim_cov)

,0,1,2,3,4
0,0.085300,0.117222,0.042409,0.009025,0.003891
1,0.117222,0.161090,0.058280,0.012403,0.005347
2,0.042409,0.058280,0.037392,0.006024,0.002585
3,0.009025,0.012403,0.006024,0.001100,0.000473
4,0.003891,0.005347,0.002585,0.000473,0.000203
